In [1]:
import numpy as np

## Preliminary

If each slot is a 32-bit number, each slab has 8 slots, and address memory natively (as bytes), how many low bits are available for tagging?

In [2]:
np.log2(4 * 8)

5.0

In [3]:
2**5 == 4 * 8 == 32

True

Each address is

In [4]:
0b100000

32

bits past the previous address. Those 5 bits are ours for tagging.

| bit | meaning |
|:----|:--------|
| 1 (lowest) | is integer |
| 2 | is floating point |
| 3 | is region |
| 4 | is enum |
| 5 | is internal |

With 64-bit numbers, there are 2 potential reserved bits.

After a conversation with myself, I concluded that we'd use whatever pointer size the underlying system has, `sizeof(size_t)`, which would usually be 64-bits. However, it's good to ensure that everything _fits_ within 32-bits.

Another thing to note, not only do most systems have 8-byte pointers (an OS choice), but most systems also have 64-byte cache lines (a hardware choice). 8 bytes × 8 slots = 64 bytes, which is perfect: a slab is a cache line. That eliminates false sharing. The slab allocator needs to be allocated (once) with `aligned_alloc`, to get `sizeof(size_t) * 8` alignment.

It needs to be possible to allocate regions with plain `malloc`. Using bit 3 as the "is region" flag means that allocations must be aligned to at least 8 bytes (64 bits). In 32-bit pointer mode, that may need to be an `aligned_alloc`, but for 64-bit pointers, it's always fine to use plain `malloc`.

## Tagging

In [5]:
class Tagged:
    def __init__(self, data):
        self._data = data

    def __repr__(self):
        return f"<Tagged{''.join(' ' + x if i % 8 == 0 else x for i, x in enumerate(format(self._data, '032b')))}>"

    @property
    def raw(self):
        return self._data

    @property
    def is_int(self):
        return bool(self._data & np.uint32(0b00000000000000000000000000000001))

    @property
    def is_float(self):
        return not bool(self._data & np.uint32(0b00000000000000000000000000000001)) and bool(self._data & np.uint32(0b00000000000000000000000000000010))

    @property
    def is_region(self):
        return not bool(self._data & np.uint32(0b00000000000000000000000000000011)) and bool(self._data & np.uint32(0b00000000000000000000000000000100))

    @property
    def is_enum(self):
        return not bool(self._data & np.uint32(0b00000000000000000000000000000111)) and bool(self._data & np.uint32(0b00000000000000000000000000001000))

    @property
    def is_internal(self):
        return not bool(self._data & np.uint32(0b00000000000000000000000000001111)) and bool(self._data & np.uint32(0b00000000000000000000000000010000))

    @property
    def is_external(self):
        return not bool(self._data & np.uint32(0b00000000000000000000000000011111))

    @classmethod
    def from_int(cls, data):
        return cls((np.int32(data).view(np.uint32) << np.uint32(1)) | np.uint32(0b00000000000000000000000000000001))

    @classmethod
    def from_float(cls, data):
        return cls((np.float32(data).view(np.uint32) & np.uint32(0b11111111111111111111111111111110)) | np.uint32(0b00000000000000000000000000000010))

    @classmethod
    def from_region(cls, data):
        return cls(np.uint32(data) | np.uint32(0b00000000000000000000000000000100))

    @classmethod
    def from_enum(cls, data):
        return cls(np.uint32(data) | np.uint32(0b00000000000000000000000000001000))

    @classmethod
    def from_internal(cls, data):
        return cls(np.uint32(data) | np.uint32(0b00000000000000000000000000010000))

    @classmethod
    def from_external(cls, data):
        return cls(np.uint32(data))

    def to_int(self):
        return ((self._data >> np.uint32(1)) | (self._data & np.uint32(0b10000000000000000000000000000000))).view(np.int32)

    def to_float(self):
        return (self._data & np.uint32(0b11111111111111111111111111111100)).view(np.float32)

    def to_region(self):
        return self._data & np.uint32(0b11111111111111111111111111111000)

    def to_enum(self):
        return self._data & np.uint32(0b11111111111111111111111111110000)

    def to_internal(self):
        return self._data & np.uint32(0b11111111111111111111111111100000)

    def to_external(self):
        return self._data

### Test tagging

In [6]:
x = Tagged.from_int(255)
x.is_int, x.is_float, x.is_region, x.is_enum, x.is_internal, x.is_external

(True, False, False, False, False, False)

In [7]:
print(bin(np.float32(0.111114495).view(np.uint32)), "(lots of 1's at the end)")
x = Tagged.from_float(0.111114495)
x.is_int, x.is_float, x.is_region, x.is_enum, x.is_internal, x.is_external

0b111101111000111000111111111111 (lots of 1's at the end)


(False, True, False, False, False, False)

In [8]:
x = Tagged.from_region(8)   # all region pointers are in increments of 4 * 2 = 8
x.is_int, x.is_float, x.is_region, x.is_enum, x.is_internal, x.is_external

(False, False, True, False, False, False)

In [9]:
x = Tagged.from_enum(16)   # enums are defined in increments of 16
x.is_int, x.is_float, x.is_region, x.is_enum, x.is_internal, x.is_external

(False, False, False, True, False, False)

In [10]:
x = Tagged.from_internal(32)   # all slab pointers are in increments of 4 * 8 = 32
x.is_int, x.is_float, x.is_region, x.is_enum, x.is_internal, x.is_external

(False, False, False, False, True, False)

In [11]:
x = Tagged.from_external(32)   # all slab pointers are in increments of 4 * 8 = 32
x.is_int, x.is_float, x.is_region, x.is_enum, x.is_internal, x.is_external

(False, False, False, False, False, True)

In [12]:
Tagged.from_int(0).to_int()

0

In [13]:
Tagged.from_int(1).to_int()

1

In [14]:
Tagged.from_int(-1).to_int()

-1

In [15]:
Tagged.from_int(2**30 - 1).to_int() == 2**30 - 1

True

In [16]:
Tagged.from_int(-2**30).to_int() == -2**30

True

In [17]:
Tagged.from_float(3.14).to_float()

3.1399994

In [18]:
Tagged.from_float(-3.14).to_float()

-3.1399994

In [19]:
Tagged.from_float(np.inf).to_float()

inf

In [20]:
Tagged.from_float(-np.inf).to_float()

-inf

In [21]:
Tagged.from_float(np.nan).to_float()

nan

## Slabs and slab allocator

In [22]:
class Slab:
    def __init__(self, pointer, data):
        self._pointer = pointer
        self._data = data

    @property
    def pointer(self):
        return self._pointer

    def __getitem__(self, i):
        return self._data[i]

    def __setitem__(self, i, x):
        self._data[i] = x

In [23]:
class SlabAllocator:
    def __init__(self, num_slabs):
        self._arena = np.zeros(num_slabs * 4 * 8, np.uint8)
        self._end = np.uint32(len(self._arena))
        self._first_free = self._end
        self._bump_free = np.uint32(0)
        
    def __repr__(self):
        return str(self._arena.view(np.uint32).reshape(-1, 8))
    
    def alloc(self):
        if self._first_free == self._end:
            if self._bump_free == self._end:
                raise RuntimeError("not enough memory")
            else:
                pointer = self._bump_free
                self._bump_free += 4 * 8
        else:
            pointer = self._first_free
            self._first_free = self._arena[self._first_free : self._first_free + 4].view(np.uint32)[0]
        
        return Slab(pointer, self._arena[pointer : pointer + (4 * 8)].view(np.uint32))
    
    def free(self, slab):
        self._arena[slab.pointer : slab.pointer + 4].view(np.uint32)[0] = self._first_free
        self._first_free = slab.pointer

### Test slab allocator

In [24]:
allocator = SlabAllocator(10)
allocator

[[0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0]]

In [25]:
one = allocator.alloc()
one[:] = 11
allocator

[[11 11 11 11 11 11 11 11]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

In [26]:
two = allocator.alloc()
two[:] = 22
allocator

[[11 11 11 11 11 11 11 11]
 [22 22 22 22 22 22 22 22]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

In [27]:
three = allocator.alloc()
three[:] = 33
allocator

[[11 11 11 11 11 11 11 11]
 [22 22 22 22 22 22 22 22]
 [33 33 33 33 33 33 33 33]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

In [28]:
allocator.free(two)
allocator

[[ 11  11  11  11  11  11  11  11]
 [320  22  22  22  22  22  22  22]
 [ 33  33  33  33  33  33  33  33]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]]

In [29]:
allocator.free(one)
allocator

[[ 32  11  11  11  11  11  11  11]
 [320  22  22  22  22  22  22  22]
 [ 33  33  33  33  33  33  33  33]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]]

In [30]:
four = allocator.alloc()
four[:] = 44
allocator

[[ 44  44  44  44  44  44  44  44]
 [320  22  22  22  22  22  22  22]
 [ 33  33  33  33  33  33  33  33]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]]

In [31]:
five = allocator.alloc()
five[:] = 55
allocator

[[44 44 44 44 44 44 44 44]
 [55 55 55 55 55 55 55 55]
 [33 33 33 33 33 33 33 33]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

In [32]:
six = allocator.alloc()
six[:] = 66
allocator

[[44 44 44 44 44 44 44 44]
 [55 55 55 55 55 55 55 55]
 [33 33 33 33 33 33 33 33]
 [66 66 66 66 66 66 66 66]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0]]

## Types of slabs

In [33]:
def is_list_slab(slab):
    return Tagged(slab[0]).is_int

def is_instance_slab(slab):
    return Tagged(slab[0]).is_external

In [34]:
allocator = SlabAllocator(10)
slab = allocator.alloc()

slab[0] = Tagged.from_int(123).raw

is_list_slab(slab)

True

See [this](https://stackoverflow.com/a/11398748/1623645) and [that](https://stackoverflow.com/a/11376759/1623645) for fast calculations of log-base-2 (equivalent of `floor(log2(x))`).

In [ ]:
def new_structure(num_header, num_slots):
    pass

In [42]:
import math

In [47]:
math.log(100, 8), math.log(100, 2)/3

(2.2146187299249087, 2.2146187299249083)

In [40]:
for x in range(1, 100):
    print(x, np.log2(x))

1 0.0
2 1.0
3 1.584962500721156
4 2.0
5 2.321928094887362
6 2.584962500721156
7 2.807354922057604
8 3.0
9 3.169925001442312
10 3.321928094887362
11 3.4594316186372973
12 3.584962500721156
13 3.700439718141092
14 3.807354922057604
15 3.9068905956085187
16 4.0
17 4.087462841250339
18 4.169925001442312
19 4.247927513443585
20 4.321928094887363
21 4.392317422778761
22 4.459431618637297
23 4.523561956057013
24 4.584962500721156
25 4.643856189774724
26 4.700439718141092
27 4.754887502163468
28 4.807354922057604
29 4.857980995127572
30 4.906890595608519
31 4.954196310386875
32 5.0
33 5.044394119358453
34 5.087462841250339
35 5.129283016944966
36 5.169925001442312
37 5.20945336562895
38 5.247927513443585
39 5.285402218862249
40 5.321928094887363
41 5.357552004618084
42 5.392317422778761
43 5.426264754702098
44 5.459431618637297
45 5.491853096329675
46 5.523561956057013
47 5.554588851677638
48 5.584962500721156
49 5.614709844115208
50 5.643856189774724
51 5.672425341971495
52 5.700439718141092
